# Notebook 12 — a neural surrogate, and whether it is needed

Level-1 code: a two-layer softmax network in numpy mapping $(\log T, i) \to R_{i\cdot}$, trained on a grid of temperatures with whole states withheld, compared with ε*, a fixed $R$ and an interpolated $R$ on the withheld states.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom
from rtedu.transport import run, emergent_by_line
from rtedu.matrix import group_of_line, MacroatomRedistribution, build_R, MatrixRedistribution, low_rank, interpolate_R, row_error
from rtedu.redistribution import EpsilonRedistribution
from rtedu.bands import band_fluxes, magnitudes, colours
rng = np.random.default_rng(rtedu.SEEDS["ch12"])
atom = five_level_atom(); nm = 1e7 * atom.lam_cm
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nu_launch = atom.nu[0] * 1.001
def spectrum(model, n, seed):
    """emergent line spectrum, band magnitudes and colours of n blue packets under a redistribution model"""
    nu, last, n_int = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau, r_out, t, model)
    m = magnitudes(band_fluxes(nu)); return emergent_by_line(last, atom.n_lines), m, colours(m), float(n_int.mean())

In [ ]:
from rtedu.surrogate import MLP, dataset, split_by_state, predict_R, features
n_g = 10; g4, _ = group_of_line(atom.nu, n_g)           # one group per line: the surrogate learns the state dependence, not the binning
T_grid = [2000.0, 2500.0, 3000.0, 3500.0, 4000.0, 5000.0, 6000.0, 7000.0, 8500.0]
held_out = [3500.0, 6000.0]                                       # entire physical states withheld
def R_at(T_, n=4000, seed=0):
    tau_ = atom.line_list(T_, n_total, t); macro = MacroatomRedistribution(atom, tau_)
    run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau_, r_out, t, macro)
    return build_R(macro.events, g4, n_g), np.bincount([g4[k] for k, _ in macro.events], minlength=n_g)
RU = [R_at(T_, seed=rtedu.SEEDS["ch12"] + i) for i, T_ in enumerate(T_grid)]; Rs = [x[0] for x in RU]; usage = [x[1] for x in RU]
train, held = split_by_state(T_grid, held_out)
print("training states:", [T_grid[i] for i in train]); print("held-out states:", [T_grid[i] for i in held])

## Level 1: the network

One hidden layer of tanh units, a softmax output, cross-entropy against the tabulated rows, plain gradient descent written out.

In [ ]:
X, Y = dataset([T_grid[i] for i in train], [Rs[i] for i in train], n_g)
def train_mlp(seed, n_hidden=24, epochs=4000, lr=0.4):
    r_ = np.random.default_rng(seed)
    W1 = r_.normal(0, 1 / np.sqrt(X.shape[1]), (X.shape[1], n_hidden)); b1 = np.zeros(n_hidden)
    W2 = r_.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, n_g)); b2 = np.zeros(n_g)
    for _ in range(epochs):
        H = np.tanh(X @ W1 + b1); z = H @ W2 + b2; z -= z.max(axis=1, keepdims=True)
        P = np.exp(z); P /= P.sum(axis=1, keepdims=True)
        dz = (P - Y) / X.shape[0]                              # softmax + cross-entropy gradient
        dW2 = H.T @ dz; db2 = dz.sum(0); dH = dz @ W2.T * (1 - H ** 2); dW1 = X.T @ dH; db1 = dH.sum(0)
        W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2
    return W1, b1, W2, b2
W1, b1, W2, b2 = train_mlp(rtedu.SEEDS["ch12"])
def R_nn(T_):
    Xq = features(T_, n_g); H = np.tanh(Xq @ W1 + b1); z = H @ W2 + b2; z -= z.max(axis=1, keepdims=True)
    P = np.exp(z); return P / P.sum(axis=1, keepdims=True)
# rtedu's class is the same network
m = MLP(X.shape[1], 24, n_g, seed=rtedu.SEEDS["ch12"]); m.fit(X, Y, epochs=4000, lr=0.4)
assert np.abs(predict_R(m, 3500.0, n_g) - R_nn(3500.0)).max() < 1e-9

## Four models on the withheld states

The error is measured through the transport: the emergent spectrum at the withheld temperature, against the macroatom there.

In [ ]:
n = 6000
def spectrum_at(T_, model, seed):
    tau_ = atom.line_list(T_, n_total, t)
    nu, last, _ = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau_, r_out, t, model)
    return emergent_by_line(last, atom.n_lines)
T_train = [T_grid[i] for i in train]; R_train = [Rs[i] for i in train]
out = {}
for T_ in held_out:
    tau_ = atom.line_list(T_, n_total, t); emis_ = atom.thermal_emissivity(T_, n_total)
    ref = spectrum_at(T_, MacroatomRedistribution(atom, tau_), rtedu.SEEDS["ch12"] + 50)
    noise = float(4 * np.sqrt(ref * (1 - ref) / n).sum())
    # eps*: the best scalar on the training states, applied here
    eps_grid = np.linspace(0, 1, 11)
    err_e = [np.mean([np.abs(spectrum_at(Tt, EpsilonRedistribution(e, atom.thermal_emissivity(Tt, n_total)), 7) - spectrum_at(Tt, MacroatomRedistribution(atom, atom.line_list(Tt, n_total, t)), 8)).sum() for Tt in T_train[::2]]) for e in eps_grid]
    eps_star = float(eps_grid[int(np.argmin(err_e))])
    models = {"eps*": EpsilonRedistribution(eps_star, emis_),
              "R fixed (4000 K)": MatrixRedistribution(Rs[T_grid.index(4000.0)], g4, emis_),
              "R interpolated": MatrixRedistribution(interpolate_R(T_, T_train, R_train), g4, emis_),
              "R neural": MatrixRedistribution(R_nn(T_), g4, emis_)}
    errs = {k: float(np.abs(spectrum_at(T_, mdl, rtedu.SEEDS["ch12"] + 60) - ref).sum()) for k, mdl in models.items()}
    w_ = usage[T_grid.index(T_)]           # usage-weighted row errors: rows that are never used do not count
    rows = {"R fixed (4000 K)": row_error(Rs[T_grid.index(4000.0)], Rs[T_grid.index(T_)], w_), "R interpolated": row_error(interpolate_R(T_, T_train, R_train), Rs[T_grid.index(T_)], w_), "R neural": row_error(R_nn(T_), Rs[T_grid.index(T_)], w_)}
    out[f"{T_:.0f}"] = dict(errs=errs, rows=rows, noise=noise, eps_star=eps_star)
    print(f"held-out T = {T_:.0f} K (eps* = {eps_star:.1f}, 4-sigma noise {noise:.3f}):", {k: round(v, 3) for k, v in errs.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, T_ in zip(axes, held_out):
    o = out[f"{T_:.0f}"]; names = list(o["errs"])
    ax.bar(names, [o["errs"][k] for k in names], color=[OI["red"], OI["orange"], OI["green"], OI["purple"]]); ax.axhline(o["noise"], color="grey", ls="--", label="4-sigma noise")
    ax.set_title(f"withheld state T = {T_:.0f} K", fontsize=9); ax.set_ylabel("spectrum error vs macroatom"); ax.tick_params(axis="x", labelsize=7); ax.legend(fontsize=7)
fig.suptitle("what complexity of effective model does the physics require? (train/test split by whole states)", fontsize=9); fig.tight_layout(); save_fig(fig, "ch12_surrogate")

In [ ]:
results.record("ch12", dict(T_grid=T_grid, held_out=held_out, n_g=n_g, n_train_states=len(train), n=n, n_hidden=24, n_weights=int(W1.size + b1.size + W2.size + b2.size), results=out))